# NeoLM — Pretraining from Scratch

Pretrain a dense decoder-only Transformer using PyTorch and Hugging Face Transformers.

## Architecture

- **Parameters:** 61.36M
- **Vocabulary:** 49,152
- **Layers:** 12
- **Hidden size:** 512
- **Attention:** Grouped-Query Attention (8 Q / 2 KV)
- **Context length:** 1,024
- **Components:** RoPE, RMSNorm, QK-Norm, SwiGLU, Tied Embeddings

## Pipeline

1. Load tokenizer and memory-mapped token streams.
2. Initialize NeoLM from scratch.
3. Configure training and evaluation.
4. Train with checkpointing.
5. Evaluate loss and perplexity.
6. Save the trained model and training metrics.

The published model is available at [NeoLM-Small-Base](https://huggingface.co/MaagDeveloper/NeoLM-Small-Base).

This notebook documents the clean pretraining pipeline. The original training run was interrupted and later extended through continued pretraining, documented separately.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    default_data_collator,
    set_seed,
)

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

from neolm.configuration_neolm import NeoLMConfig
from neolm.modeling_neolm import NeoLMForCausalLM
import math
from torch.utils.data import Subset
from transformers.trainer_utils import get_last_checkpoint
import matplotlib.pyplot as plt
import copy
import gc
import tempfile

In [ ]:
# Paths
MODEL_DIR = ROOT / "artifacts/model"
DB_DIR = ROOT / "artifacts/pretrain_db"
CHECKPOINT_DIR = ROOT / "artifacts/checkpoints"

# Reproducibility
SEED = 1337
set_seed(SEED)

# Data
BLOCK_SIZE = 1024
STRIDE = BLOCK_SIZE - 1
TOKEN_DTYPE = np.uint16

EXPECTED_TOKENS = {
    "train": 1_147_218_931,
    "val": 3_084_132,
    "test": 12_094_582,
}

# Hardware
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_bf16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)
use_fp16 = device.type == "cuda" and not use_bf16

# Validate required files
assert MODEL_DIR.is_dir(), "Tokenizer directory not found"

for split, count in EXPECTED_TOKENS.items():
    path = DB_DIR / f"{split}.bin"

    assert path.is_file(), f"Missing: {path}"
    assert path.stat().st_size == count * np.dtype(TOKEN_DTYPE).itemsize

print("=" * 60)
print("NEOLM — PRETRAINING CONFIGURATION")
print("=" * 60)
print(f"Device:          {device}")
print(f"BF16:            {use_bf16}")
print(f"FP16:            {use_fp16}")
print(f"Block size:      {BLOCK_SIZE:,}")
print(f"Stride:          {STRIDE:,}")
print(f"Train tokens:    {EXPECTED_TOKENS['train']:,}")
print(f"Model directory: {MODEL_DIR}")
print("=" * 60)

In [ ]:
class MemmapBlocksDataset(Dataset):
    def __init__(self, path, block_size=1024, stride=1023):
        self.tokens = np.memmap(
            path,
            dtype=np.uint16,
            mode="r",
        )

        self.block_size = block_size
        self.stride = stride

        assert len(self.tokens) >= block_size

        self.n_blocks = (
            (len(self.tokens) - block_size) // stride + 1
        )

    def __len__(self):
        return self.n_blocks

    def __getitem__(self, idx):
        start = idx * self.stride

        block = self.tokens[start:start + self.block_size]

        ids = torch.from_numpy(block.astype(np.int64))

        return {
            "input_ids": ids,
            "labels": ids.clone(),
        }


train_dataset = MemmapBlocksDataset(
    DB_DIR / "train.bin", BLOCK_SIZE, STRIDE
)

eval_dataset = MemmapBlocksDataset(
    DB_DIR / "val.bin", BLOCK_SIZE, STRIDE
)

test_dataset = MemmapBlocksDataset(
    DB_DIR / "test.bin", BLOCK_SIZE, STRIDE
)

print("=" * 60)
print("NEOLM — MEMMAP DATASETS")
print("=" * 60)

print(f"Train blocks:      {len(train_dataset):,}")
print(f"Validation blocks: {len(eval_dataset):,}")
print(f"Test blocks:       {len(test_dataset):,}")
print(f"Sample shape:      {tuple(train_dataset[0]['input_ids'].shape)}")

assert train_dataset[0]["input_ids"].shape == (BLOCK_SIZE,)
assert torch.equal(
    train_dataset[0]["input_ids"],
    train_dataset[0]["labels"],
)

print("=" * 60)
print("MEMMAP DATASETS VERIFIED ✅")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    use_fast=True,
    trust_remote_code=True,
)

# Register custom architecture for Hugging Face
NeoLMConfig.register_for_auto_class()
NeoLMForCausalLM.register_for_auto_class("AutoModelForCausalLM")

# Model configuration
config = NeoLMConfig(
    vocab_size=len(tokenizer),
    max_position_embeddings=BLOCK_SIZE,
    hidden_size=512,
    num_hidden_layers=12,
    num_attention_heads=8,
    num_key_value_heads=2,
    intermediate_size=1536,
    dropout=0.1,
    tie_word_embeddings=True,
    qk_norm=True,
    scale_residual_init=True,
    rope_base=10_000,
    use_cache=False,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

# Initialize from scratch
model = NeoLMForCausalLM(config).to(device)

params = sum(p.numel() for p in model.parameters())

assert params == 61_356_032

print("=" * 60)
print("NEOLM — MODEL INITIALIZATION")
print("=" * 60)
print(f"Parameters:    {params:,} ({params / 1e6:.2f}M)")
print(f"Vocabulary:    {config.vocab_size:,}")
print(f"Layers:        {config.num_hidden_layers}")
print(f"Hidden size:   {config.hidden_size}")
print(f"Attention:     {config.num_attention_heads}Q / {config.num_key_value_heads}KV")
print(f"Context:       {config.max_position_embeddings:,}")
print(f"Device:        {next(model.parameters()).device}")
print("=" * 60)
print("MODEL INITIALIZED ✅")


### Model Sanity Check

Before training, verify that the randomly initialized model:

- Computes a finite loss during the forward pass.
- Produces valid, non-zero gradients during backpropagation.
- Correctly ties input embeddings to the output projection.

This is a lightweight verification, not a training run or model quality evaluation.


In [ ]:
model.train()
model.zero_grad(set_to_none=True)

# Small batch for verification
input_ids = train_dataset[0]["input_ids"][:128]
input_ids = input_ids.unsqueeze(0).to(device)

# Forward
outputs = model(
    input_ids=input_ids,
    labels=input_ids,
)

loss = outputs.loss

assert torch.isfinite(loss), "Invalid loss"

# Backward
loss.backward()

grad = model.model.blocks[0].attn.qkv.weight.grad

assert grad is not None
assert torch.isfinite(grad).all()
assert grad.norm().item() > 0

# Weight tying
tied = (
    model.model.token_emb.weight.data_ptr()
    == model.lm_head.weight.data_ptr()
)

assert tied, "Embedding weights are not tied"

print("=" * 60)
print("NEOLM — FORWARD & BACKWARD VERIFICATION")
print("=" * 60)
print(f"Input shape:    {tuple(input_ids.shape)}")
print(f"Loss:           {loss.item():.4f}")
print(f"Gradient norm:  {grad.norm().item():.6f}")
print(f"Tied weights:   {tied}")
print("=" * 60)
print("FORWARD & BACKWARD PASSED ✅")

# Cleanup
model.zero_grad(set_to_none=True)
del outputs, loss, grad, input_ids

In [ ]:
# Training
NUM_EPOCHS = 2
BATCH_SIZE = 4
GRAD_ACCUM = 16

EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM

STEPS_PER_EPOCH = math.ceil(
    len(train_dataset) / EFFECTIVE_BATCH_SIZE
)

TOTAL_STEPS = STEPS_PER_EPOCH * NUM_EPOCHS
WARMUP_STEPS = math.ceil(TOTAL_STEPS * 0.03)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    # Training
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Optimizer & Scheduler
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Precision
    bf16=use_bf16,
    fp16=use_fp16,

    # Evaluation & Checkpoints
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=5,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=1000,
    logging_first_step=True,
    report_to="none",

    # Reproducibility
    seed=SEED,
    data_seed=SEED,

    # Data loading
    dataloader_num_workers=2,
    dataloader_pin_memory=device.type == "cuda",

    label_names=["labels"],
)

print("=" * 60)
print("NEOLM — TRAINING CONFIGURATION")
print("=" * 60)
print(f"Epochs:           {NUM_EPOCHS}")
print(f"Micro-batch:      {BATCH_SIZE}")
print(f"Grad accumulation:{GRAD_ACCUM:>5}")
print(f"Effective batch:  {EFFECTIVE_BATCH_SIZE}")
print(f"Steps/epoch:      {STEPS_PER_EPOCH:,}")
print(f"Total steps:      {TOTAL_STEPS:,}")
print(f"Warmup steps:     {WARMUP_STEPS:,}")
print(f"Learning rate:    {training_args.learning_rate}")
print(f"Precision:        {'BF16' if use_bf16 else 'FP16' if use_fp16 else 'FP32'}")
print(f"Checkpoints:      {CHECKPOINT_DIR}")
print("=" * 60)
print("TRAINING CONFIG READY ✅")

In [ ]:
eval_indices = np.linspace(
    0,
    len(eval_dataset) - 1,
    num=256,
    dtype=np.int64,
).tolist()

quick_eval_dataset = Subset(eval_dataset, eval_indices)

assert len(quick_eval_dataset) == 256

print("=" * 60)
print("NEOLM — VALIDATION CONFIGURATION")
print("=" * 60)
print(f"Quick validation blocks: {len(quick_eval_dataset):,}")
print(f"Full validation blocks:  {len(eval_dataset):,}")
print(f"Evaluation interval:     {training_args.eval_steps:,} steps")
print("=" * 60)
print("VALIDATION DATASET READY ✅")

In [ ]:
def collate_fn(features):
    batch = default_data_collator(features)

    batch["input_ids"] = batch["input_ids"].long()
    batch["labels"] = batch["labels"].long()

    return batch


class PerplexityTrainer(Trainer):
    def log(self, logs, *args, **kwargs):
        logs = dict(logs)

        for key, loss in list(logs.items()):
            if key == "loss":
                prefix = "train"
            elif key.endswith("_loss"):
                prefix = key.removesuffix("_loss")
            else:
                continue

            if isinstance(loss, (int, float)):
                logs[f"{prefix}_ppl"] = (
                    round(math.exp(loss), 2)
                    if math.isfinite(loss) and loss < 700
                    else float("inf")
                )

        super().log(logs, *args, **kwargs)


trainer = PerplexityTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=quick_eval_dataset,
    data_collator=collate_fn,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=15,
            early_stopping_threshold=0.001,
        )
    ],
)

# Model computes its own mean loss
trainer.model_accepts_loss_kwargs = False

print("=" * 60)
print("NEOLM — TRAINER CONFIGURATION")
print("=" * 60)
print(f"Train blocks:       {len(trainer.train_dataset):,}")
print(f"Validation blocks:  {len(trainer.eval_dataset):,}")
print(f"Effective batch:    {EFFECTIVE_BATCH_SIZE}")
print(f"Evaluation steps:   {training_args.eval_steps:,}")
print(f"Checkpoint steps:   {training_args.save_steps:,}")
print("Early stopping:     15 evaluations")
print(f"Best model metric:  {training_args.metric_for_best_model}")
print(f"Label names:        {trainer.label_names}")
print("=" * 60)
print("TRAINER READY ✅")


### Pilot Training — Sanity Check

Run two optimizer steps to verify the complete training pipeline and measure GPU memory usage.

This is a short hardware and training sanity check, not a model quality evaluation. The pilot uses a separate randomly initialized model.


In [ ]:
pilot_args = copy.deepcopy(training_args)

pilot_dir = tempfile.TemporaryDirectory(prefix="neolm_pilot_")
pilot_args.output_dir = pilot_dir.name
pilot_args.max_steps = 2
pilot_args.eval_strategy = "no"
pilot_args.save_strategy = "no"
pilot_args.load_best_model_at_end = False
pilot_args.logging_steps = 1

# Separate model to preserve the original initialization
pilot_model = NeoLMForCausalLM(config).to(device)

pilot_trainer = PerplexityTrainer(
    model=pilot_model,
    args=pilot_args,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    data_collator=default_data_collator,
)

pilot_trainer.model_accepts_loss_kwargs = False

torch.cuda.reset_peak_memory_stats()

print("=" * 60)
print("NEOLM — PILOT TRAINING")
print("=" * 60)
print(f"Optimizer steps:     {pilot_args.max_steps}")
print(f"Micro-batch:         {BATCH_SIZE}")
print(f"Gradient accumulation: {GRAD_ACCUM}")
print(f"Effective batch:     {EFFECTIVE_BATCH_SIZE}")
print("=" * 60)

result = pilot_trainer.train()

print(f"\n{'=' * 60}")
print("PILOT TRAINING COMPLETE")
print("=" * 60)
print(f"Training loss: {result.training_loss:.4f}")
print(f"Peak VRAM:     {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
print(f"Runtime:       {result.metrics['train_runtime']:.1f}s")
print("=" * 60)

# Cleanup
del pilot_trainer, pilot_model
gc.collect()
torch.cuda.empty_cache()

pilot_dir.cleanup()

In [ ]:
metrics = trainer.evaluate(
    eval_dataset=Subset(eval_dataset, [0, 1])
)

assert "eval_loss" in metrics, "Evaluation loss is missing!"

print("Evaluation loss:", metrics["eval_loss"])
print("Early stopping metric available ✅")


### Full Pretraining

Train NeoLM from scratch using the configured Hugging Face Trainer.

Training supports checkpoint resumption, periodic validation, early stopping, and automatic loading of the best checkpoint.

**Note:** Full training is disabled by default because the released NeoLM model has already been trained. Enable it only when starting a new experiment.


In [ ]:
RESUME_TRAINING = True

checkpoint = (
    get_last_checkpoint(str(CHECKPOINT_DIR))
    if RESUME_TRAINING and CHECKPOINT_DIR.exists()
    else None
)

print("=" * 60)
print("NEOLM — PRETRAINING")
print("=" * 60)
print(f"Total steps: {TOTAL_STEPS:,}")
print(f"Resume from: {checkpoint or 'Scratch'}")
print("=" * 60)

train_result = trainer.train(
    resume_from_checkpoint=checkpoint
)

print(f"\n{'=' * 60}")
print("PRETRAINING COMPLETE")
print("=" * 60)
print(f"Training loss:   {train_result.training_loss:.4f}")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best eval loss:  {trainer.state.best_metric}")
print("=" * 60)


### Final Test Evaluation

Evaluate the fully trained model on the complete held-out test split.

Report test loss and perplexity. This step should only run after training is complete and the selected model checkpoint has been loaded.

**Note:** Disabled by default. The short pilot run is not a meaningful model quality evaluation.


In [ ]:
test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)

test_loss = test_metrics["test_loss"]
test_ppl = (
    math.exp(test_loss)
    if math.isfinite(test_loss) and test_loss < 700
    else float("inf")
)

print(f"\n{'=' * 60}")
print("NEOLM — FINAL TEST EVALUATION")
print(f"{'=' * 60}")
print(f"Test blocks:      {len(test_dataset):,}")
print(f"Test loss:        {test_loss:.4f}")
print(f"Test perplexity:  {test_ppl:.4f}")
print(f"{'=' * 60}")


### Save Best Model

Save the best validation checkpoint alongside the tokenizer and model configuration.

Run only after training has completed and the best checkpoint has been restored.


In [ ]:
assert trainer.state.best_model_checkpoint is not None, (
    "No best checkpoint available."
)

# Enable KV Cache for inference
trainer.model.config.use_cache = True
trainer.model.generation_config.use_cache = True

# Save model + tokenizer (processing_class)
trainer.save_model(str(MODEL_DIR))

# Save generation configuration
trainer.model.generation_config.save_pretrained(
    str(MODEL_DIR)
)

print(f"\n{'=' * 60}")
print("NEOLM — BEST MODEL SAVED")
print(f"{'=' * 60}")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best eval loss:  {trainer.state.best_metric:.4f}")
print(f"Model directory: {MODEL_DIR.resolve()}")
print(f"KV Cache:        {trainer.model.config.use_cache}")
print(f"{'=' * 60}")



### Training Curves

Visualize training and validation loss across optimizer steps using the Trainer's recorded history.

Short test runs verify logging functionality but are not sufficient to assess model convergence.


In [ ]:
logs = trainer.state.log_history

train_logs = [e for e in logs if "loss" in e]
eval_logs = [e for e in logs if "eval_loss" in e]

plt.figure(figsize=(10, 5))

if train_logs:
    plt.plot(
        [e["step"] for e in train_logs],
        [e["loss"] for e in train_logs],
        label="Training Loss",
    )

if eval_logs:
    plt.plot(
        [e["step"] for e in eval_logs],
        [e["eval_loss"] for e in eval_logs],
        label="Validation Loss",
        marker="o",
    )

plt.xlabel("Optimizer Steps")
plt.ylabel("Loss")
plt.title("NeoLM — Pretraining Loss")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()